> ## SOLUTION / ANSWER KEY &mdash; Lab 4.12
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-12-capstone-adapt-a-model.ipynb`](../lab-12-capstone-adapt-a-model.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 4.12 &mdash; Capstone: Fine-tune a Real Model to a New Task

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Pre-trained Models & Fine-tuning**

### What you'll do
- Apply the whole Module 4 pipeline to a NEW task (topic: sports vs tech)
- Tokenize, split, and really fine-tune a real model end-to-end
- Prove a before -> after improvement and predict on unseen text

> **How this lab works (near-real):** these labs use **real Hugging Face Transformers** locally on CPU &mdash; a real pretrained sentiment model, a real tokenizer, and (the headline) a **real fine-tune** you run yourself. Read the **Concept**, fill the real `___` blanks in **Build it** (real model / tokenizer / training-loop calls), **Run it for real** to see the **actual model output** (including the real **before &rarr; after** fine-tune numbers), note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real results you can read. The genuine maths (softmax, precision/recall) you still compute **by hand** &mdash; real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased-finetuned-sst-2-english` (out-of-the-box sentiment + logits), `distilbert-base-uncased` (tokenizer), `prajjwal1/bert-tiny` (frozen features **and** the model you fine-tune). First use downloads the weights (needs network), then they are cached. An optional hosted comparison uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 4 slides &mdash; Fine-tuning end-to-end](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 4 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (optional hosted compare)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-04-12")
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
Capstone: the full fine-tuning workflow on a **new** task &mdash; classify a sentence as **sports (0)**
vs **tech (1)**. Same recipe as sentiment, different labels: load a real model with a fresh head,
tokenize, split, run a real training loop, and measure held-out accuracy **before vs after**. This is
exactly how you adapt a pre-trained model to *your* problem.

## Build it
Complete the fine-tune loop and the evaluation for the new task.

In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def load():
    tok = AutoTokenizer.from_pretrained("prajjwal1/bert-tiny")
    torch.manual_seed(0)
    model = AutoModelForSequenceClassification.from_pretrained("prajjwal1/bert-tiny", num_labels=2)
    return tok, model

def evaluate(model, tok, texts, y):
    model.eval()
    enc = tok(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        pred = model(**enc).logits.argmax(-1).numpy()
    return float((pred == np.array(y)).mean()), pred

def fine_tune(model, tok, texts, y, steps=50, lr=5e-3):
    enc = tok(texts, padding=True, truncation=True, return_tensors="pt"); yt = torch.tensor(y)
    opt = torch.optim.AdamW(model.parameters(), lr=lr); model.train()
    for _ in range(steps):
        opt.zero_grad(); out = model(**enc, labels=yt)
        out.loss.backward(); opt.step()
    return float(out.loss)

## Run it for real
Run the whole pipeline on the new sports-vs-tech task.

In [ ]:
try:
    # A new tiny task: sports (0) vs tech (1). Keywords recur so the model generalises.
    TASK = [
        ("the team scored a goal to win the game", 0), ("our team won the match with a late goal", 0),
        ("the player scored twice in the game", 0), ("a great goal helped the team win the match", 0),
        ("the coach and team celebrated the win", 0), ("the player passed the ball and scored a goal", 0),
        ("the team lost the game by one goal", 0), ("the striker scored the winning goal", 0),
        ("the new app runs on a fast chip", 1), ("the software update improved the app", 1),
        ("the app stores its data in the cloud", 1), ("a new chip makes the computer software faster", 1),
        ("the cloud server runs the data software", 1), ("the app uses data and a fast chip", 1),
        ("the computer software had a data bug", 1), ("the developer shipped the new software app", 1),
    ]
    X = [t for t, y in TASK]; Y = [y for t, y in TASK]
    from sklearn.model_selection import train_test_split
    Xtr, Xval, ytr, yval = train_test_split(X, Y, test_size=0.3, random_state=0, stratify=Y)

    tok, model = load()
    before, _ = evaluate(model, tok, Xval, yval)
    loss = fine_tune(model, tok, Xtr, ytr)
    after, _ = evaluate(model, tok, Xval, yval)
    print(f"NEW TASK sports-vs-tech | BEFORE val acc = {before:.3f}  ->  AFTER (loss {loss:.3f}) = {after:.3f}  ({after-before:+.3f})")

    for s in ["the striker scored a header for the team", "the gpu accelerates the software app"]:
        enc = tok([s], return_tensors="pt")
        with torch.no_grad(): p = int(model(**enc).logits.argmax(-1))
        print(f"  pred={p} ({'tech' if p==1 else 'sports'})  <-  {s}")
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

## What to notice
- Same loop as sentiment, **new labels** &mdash; that is the whole point: fine-tuning adapts a pre-trained model to *any* small task you have labels for.
- The **before &rarr; after** jump on held-out data proves the model learned the new task, not the training rows.
- You changed nothing but the data and labels &mdash; the pretrained backbone did the heavy lifting.

## Your turn (open task &mdash; no grader)
Invent your **own** two-class task (spam vs ham, question vs statement, urgent vs not) with a
dozen labelled sentences and run the exact same pipeline. Does it converge? A "good" answer: you can
name every stage &mdash; tokenize, split, fine-tune, evaluate &mdash; and say what would break if you
skipped the held-out split.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
try:
    # YOUR OWN new two-class task: spam (1) vs ham (0). Keywords recur so it generalises.
    from sklearn.model_selection import train_test_split
    MINE = [
        ("win a free prize now click here", 1), ("claim your free cash prize today", 1),
        ("free money click this link right now", 1), ("you won a free gift claim it now", 1),
        ("congrats free voucher click now to win", 1), ("free free free click to win cash", 1),
        ("lunch meeting moved to noon today", 0), ("can you review the report today", 0),
        ("the meeting notes are attached here", 0), ("please send me the project report", 0),
        ("see you at the team meeting today", 0), ("the report review is due tomorrow", 0),
    ]
    X = [t for t, y in MINE]; Y = [y for t, y in MINE]
    Xtr, Xval, ytr, yval = train_test_split(X, Y, test_size=0.3, random_state=0, stratify=Y)
    tok, model = load()                    # tokenize + fresh head
    before, _ = evaluate(model, tok, Xval, yval)
    fine_tune(model, tok, Xtr, ytr)        # real training loop
    after, _ = evaluate(model, tok, Xval, yval)
    print(f"spam-vs-ham | before={before:.3f} -> after={after:.3f}  ({after-before:+.3f})")
    print("Stages: tokenize -> split -> fine-tune -> evaluate. Skip the held-out split and you'd measure memorisation, not learning.")
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

---
### Key takeaway
You adapted a **real** pre-trained model to a brand-new task end-to-end. That is Module 4 in one move: stand on a pretrained model, fine-tune a small head-and-body, measure honestly, ship. Next: Day 3 &mdash; agents.

[&#8592; All Module 4 labs](./index.html) &nbsp;&middot;&nbsp; [Module 4 slides](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>